# PSSN Week 2 — Generate semantic-state representations

Run in a Colab GPU runtime after notebooks 06 and 07. This processes only the W2-004 Week-3-ready states. It loads Qwen3-4B once, runs one causal forward pass per trace, mean-pools final-layer hidden states over each state's tokenizer-offset span, and saves vectors separately from state text. On a T4 this uses FP16 model weights and FP32 output vectors. If the GitHub checkout lacks the new W2-005 modules, upload the listed source files when prompted.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY_URL = "https://github.com/Rami2212/PSSN-LLM-Reasoning.git"
REPOSITORY_DIR = Path("/content/PSSN-LLM-Reasoning")
if not (REPOSITORY_DIR / "requirements.txt").exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
os.chdir(REPOSITORY_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
from google.colab import drive
drive.mount("/content/drive")
print("Repository:", Path.cwd())

In [ ]:
try:
    from src.models.hidden_states import extract_span_hidden_vectors
    from src.semantic_states.representations import extract_semantic_state_representations
    print("W2-005 representation utilities are available in the checkout.")
except ImportError:
    from google.colab import files
    import re
    uploaded = files.upload()
    destinations = {"hidden_states.py": Path("src/models/hidden_states.py")}
    semantic_sources = {"__init__.py", "schema.py", "segmenter.py", "tokenizer_utils.py", "validation.py", "representations.py"}
    normalized_uploads = {}
    for uploaded_name, content in uploaded.items():
        canonical_name = re.sub(r" \(\d+\)(?=\.py$)", "", Path(uploaded_name).name)
        if canonical_name in semantic_sources or canonical_name in destinations:
            normalized_uploads[canonical_name] = content
    required = semantic_sources | set(destinations)
    missing = required - set(normalized_uploads)
    if missing:
        raise ValueError(f"Upload the seven listed source files; missing: {sorted(missing)}")
    semantic_dir = Path("src/semantic_states")
    semantic_dir.mkdir(parents=True, exist_ok=True)
    for filename in semantic_sources:
        (semantic_dir / filename).write_bytes(normalized_uploads[filename])
    for filename, destination in destinations.items():
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(normalized_uploads[filename])
    from src.models.hidden_states import extract_span_hidden_vectors
    from src.semantic_states.representations import extract_semantic_state_representations
    print("Uploaded the current W2-005 implementation and dependencies.")

In [ ]:
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import torch

from src.utils.environment import collect_environment_info, load_qwen_model

MODEL_ID = "Qwen/Qwen3-4B"
SELECTED_LAYER = -1
POOLING_METHOD = "mean"
ARTIFACT_ROOT = Path("/content/drive/MyDrive/PSSN2/artifacts/expanded_baseline")
environment = collect_environment_info()
if not environment["cuda_available"]:
    raise RuntimeError("Select a Colab GPU runtime before loading Qwen3-4B.")
if environment["gpus"][0]["compute_capability"] == "7.5" and environment["selected_dtype"] != "float16":
    raise RuntimeError("A Tesla T4 must use FP16; sync src/utils/environment.py and restart the runtime.")
print(json.dumps(environment, indent=2))
tokenizer, model, precision = load_qwen_model(model_id=MODEL_ID, require_cuda=True)
model.eval()
print(f"Loaded {MODEL_ID} using {precision} precision on {model.device}.")

candidate_files = sorted(ARTIFACT_ROOT.glob("*/semantic_state_segmentation/*/quality_validation/*/week3_ready_states.jsonl"))
if not candidate_files:
    raise FileNotFoundError(f"No W2-004 week3_ready_states.jsonl found under {ARTIFACT_ROOT}; run notebook 07 first.")
week3_path = candidate_files[-1]
run_dir = week3_path.parents[4]
source_path = run_dir / "segmentation_ready_traces.jsonl"
if not source_path.is_file():
    raise FileNotFoundError(f"Source trace file is missing: {source_path}")
week3_states = [json.loads(line) for line in week3_path.read_text(encoding="utf-8").splitlines() if line.strip()]
source_records = [json.loads(line) for line in source_path.read_text(encoding="utf-8").splitlines() if line.strip()]
source_by_trace = {f"{row.get('run_id')}:{row.get('problem_id')}": row for row in source_records}
groups = {}
for state in week3_states:
    groups.setdefault(state["trace_id"], []).append(state)
print(f"Valid input: {len(groups)} traces and {len(week3_states)} states from {week3_path}")

vector_chunks, index_records = [], []
representation_config = None
for trace_number, (trace_id, trace_states) in enumerate(groups.items(), start=1):
    source = source_by_trace.get(trace_id)
    if source is None:
        raise KeyError(f"No baseline trace found for {trace_id}")
    batch = extract_semantic_state_representations(
        model, tokenizer, prompt=source["prompt"], reasoning_trace=source["reasoning_trace"],
        states=trace_states, model_id=MODEL_ID, layer=SELECTED_LAYER, pooling=POOLING_METHOD,
    )
    vector_offset = sum(chunk.shape[0] for chunk in vector_chunks)
    vector_chunks.append(batch.vectors)
    for index_record in batch.index_records:
        index_record["vector_index"] += vector_offset
        index_records.append(index_record)
    if representation_config is None:
        representation_config = dict(batch.configuration)
    if trace_number % 10 == 0 or trace_number == len(groups):
        print(f"Processed {trace_number}/{len(groups)} traces")

vectors = np.concatenate(vector_chunks, axis=0).astype(np.float32, copy=False)
if vectors.shape[0] != len(index_records) or len(index_records) != len(week3_states):
    raise RuntimeError("Vector rows and state index records do not align.")
representation_config.update({
    "selected_layer": SELECTED_LAYER,
    "pooling": POOLING_METHOD,
    "precision": precision,
    "vector_dtype": str(vectors.dtype),
    "vector_shape": list(vectors.shape),
    "state_count": len(week3_states),
    "trace_count": len(groups),
    "source_week3_ready_file": str(week3_path),
    "repository_commit": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() or None,
    "source_code_sha256": {
        "hidden_states.py": hashlib.sha256(Path("src/models/hidden_states.py").read_bytes()).hexdigest(),
        "representations.py": hashlib.sha256(Path("src/semantic_states/representations.py").read_bytes()).hexdigest(),
        "schema.py": hashlib.sha256(Path("src/semantic_states/schema.py").read_bytes()).hexdigest(),
    },
    "runtime_environment": environment,
    "torch_version": torch.__version__,
})

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = week3_path.parent / "representations" / stamp
output_dir.mkdir(parents=True, exist_ok=False)
vectors_path = output_dir / "state_vectors.npy"
index_path = output_dir / "representation_index.jsonl"
config_path = output_dir / "representation_config.json"
np.save(vectors_path, vectors, allow_pickle=False)
index_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in index_records), encoding="utf-8")
representation_config["created_at_utc"] = datetime.now(timezone.utc).isoformat()
representation_config["output_files"] = {"vectors": str(vectors_path), "index": str(index_path)}
config_path.write_text(json.dumps(representation_config, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved vectors:", vectors.shape, "to", vectors_path)
print("Representation config:", json.dumps(representation_config, indent=2))

In [ ]:
print("First representation links:")
for row in index_records[:8]:
    print({key: row[key] for key in ("vector_index", "problem_id", "state_id", "token_count", "pooled_token_count", "hidden_size")})
loaded = np.load(vectors_path, mmap_mode="r", allow_pickle=False)
assert loaded.shape == vectors.shape
print("Verified separately stored vector matrix:", loaded.shape, loaded.dtype)